In [ ]:
import requests
import json
from IPython.display import display, Markdown
from openai import OpenAI

In [ ]:
# Initialize the OpenAI client pointing to the local Ollama instance
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama', # required, but unused
)

In [ ]:
def fetch_cve_details(cve_id: str):
    """
    Fetches CVE details from the official NIST NVD API.
    """
    # Using the NVD API v2
    url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"
    
    # NVD recommends adding a User-Agent
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) OS/SDL-Research"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if not data.get("vulnerabilities"):
            return None, "CVE not found or API limits exceeded."
            
        # Parse the nested JSON structure
        cve_data = data["vulnerabilities"][0]["cve"]
        
        # 1. Get Description
        descriptions = cve_data.get("descriptions", [])
        english_desc = next((desc["value"] for desc in descriptions if desc["lang"] == "en"), "No description available.")
        
        # 2. Get Metrics (try CVSS 3.1, fallback to 3.0 or 2.0)
        metrics = cve_data.get("metrics", {})
        cvss_data = None
        
        if "cvssMetricV31" in metrics:
            cvss_data = metrics["cvssMetricV31"][0]["cvssData"]
        elif "cvssMetricV30" in metrics:
            cvss_data = metrics["cvssMetricV30"][0]["cvssData"]
        elif "cvssMetricV2" in metrics:
            cvss_data = metrics["cvssMetricV2"][0]["cvssData"]
            
        score = cvss_data.get("baseScore", "Unknown") if cvss_data else "Unknown"
        severity = cvss_data.get("baseSeverity", "Unknown") if cvss_data else "Unknown"
        
        cve_info = f"**{cve_id}**\n**Severity:** {severity} (Score: {score})\n**Description:** {english_desc}"
        return cve_info, None
        
    except Exception as e:
        return None, f"Error fetching CVE: {e}"

In [ ]:
def generate_sdl_summary(cve_id: str, model_name: str = "gemma2:2b"):
    """
    Fetches a CVE, passes it to a local LLM, and formats it for enterprise SDL usage.
    """
    print(f"Fetching data for {cve_id}...")
    cve_info, error = fetch_cve_details(cve_id)
    
    if error:
        display(Markdown(f"**Error:** {error}"))
        return
        
    print(f"Data retrieved! Generating SDL Report using {model_name}...\n")
    
    # The Prompt to structure the output for Corporate SDL
    system_prompt = """You are an Expert Application Security Engineer and DevSecOps Architect. 
Your job is to take raw CVE data and translate it into a highly actionable, commercially usable 
Secure Development Lifecycle (SDL) summary. 
Please format your response using the following Markdown structure:
1. **Executive Summary:** A 1-2 sentence high-level explanation for management.
2. **Technical Threat:** How a threat actor could exploit this (keep it clear and architectural).
3. **Business Impact:** What happens if this is exploited in production.
4. **Developer Remediation Strategy:** Exactly what developers must do to patch or mitigate this in their codebase/infrastructure.
5. **SDL Integration:** How to prevent this in the future (e.g., specific SAST checks, DAST, dependency scanning, or code review guidelines).
"""
    user_prompt = f"Please analyze this vulnerability and generate the SDL report:\n{cve_info}"

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            stream=False,
            extra_body={
                "options": {
                   "num_gpu": -1
                }
            }
        )
        
        response_text = response.choices[0].message.content
        display(Markdown(response_text))
        
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
# ==========================================
# EXAMPLE USAGE
# ==========================================
generate_sdl_summary("CVE-2026-0398")
# Another Example: A high-profile OpenSSL vulnerability
# generate_sdl_summary("CVE-2014-0160", "llama3.2")

In [ ]:
from cve_analyzer import CVESecurityAnalyzer
from IPython.display import display, Markdown
# Initialize the analyzer (you can swap "llama3.2" for "gemma2:2b" or any other model you pulled)
analyzer = CVESecurityAnalyzer(model_name="gemma2:2b")
# Analyze a vulnerability! (Let's use the infamous Log4Shell as an example)
report = analyzer.generate_sdl_report("CVE-2026-0398")
# Display the final structured SDL Report natively in Jupyter
display(Markdown(report))